# Compare saved runs (no camera needed)

This loads the runs that `troubleshoot.ipynb` saved and rebuilds every plot and the comparison table, for group-meeting figures. PNGs go to `Desktop\HOI_freshstaret\data\processed\troubleshoot_figures\`. Unreadable files, such as one still being written, are skipped with a note.

`SPOT` is the beam position on the sensor. `HIT_THR` is the photon threshold in ADU above pedestal. Both default to the 20 Oct values, and section 3 can recompute them from a wide-gate pair.

In [ ]:
import sys, importlib
sys.path.insert(0, r"C:\Users\ReschLab\Single-Photon-Imaging")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
import hoi_troubleshoot as ht
importlib.reload(ht)
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 250)
SPOT, HIT_THR = (501, 454), 200

## 1. Load runs

`contains` filters by file name, for example a date such as `"20261020"` or a label such as `"ladder"`. Leave it empty to load everything.

In [ ]:
runs = ht.load_runs(ht.DATA_DIR, contains="")
print(f"{len(runs)} runs in {ht.DATA_DIR}")
listing = pd.DataFrame([{k: ht.run_metrics(r)[k] for k in
                         ("started", "label", "kind", "gain", "gate_delay_ns", "gate_width_ns",
                          "accumulations", "n_readouts", "trigger", "photon_mode", "s_per_readout",
                          "min_gates_per_s")}
                        for r in runs])
display(listing)

## 2. Choose signal and dark pairs

`ht.auto_pair` gives every non-dark run the most recent matching dark. To choose by hand, `by_label` returns the most recent run with exactly that label.

In [ ]:
def by_label(label):
    hits = [r for r in runs if r["meta"]["settings"]["label"] == label]
    if not hits:
        raise KeyError(f"no run labelled {label!r}")
    return hits[-1]

pairs = ht.auto_pair(runs)
for r, d in pairs:
    print(f"{r['meta']['settings']['label']:35s} <- dark: {d['meta']['settings']['label'] if d else 'NONE'}")

## 3. Wide-gate figures: spot, threshold, full frame

Pick the wide-gate signal and dark. This recomputes `SPOT` and `HIT_THR` from them, then draws the six-panel figure, the full-frame check, the threshold sweep and the Wiener view. Panels are explained in `troubleshoot.ipynb` section 8.

In [ ]:
SIGNAL, DARK = by_label("A_wide_gate"), by_label("A_wide_gate_blocked")
SPOT = ht.find_spot(SIGNAL, DARK)
thr_table, HIT_THR = ht.threshold_sweep(SIGNAL, DARK, spot=SPOT, save_as="A_threshold_sweep")
print("SPOT", SPOT, " HIT_THR", HIT_THR)
ht.plot_run(SIGNAL, DARK, threshold=HIT_THR, spot=SPOT, save_as="A_wide_gate_overview")
ht.plot_full_frame(SIGNAL, DARK, save_as="A_full_frame_check")
ht.plot_raw_vs_accumulated(SIGNAL, DARK, save_as="A_wide_gate_raw_vs_sum")
ht.plot_wiener(SIGNAL, DARK, save_as="A_wide_gate_wiener")

## 4. Summary table and side-by-side images

One row per signal run: settings, timing and the trigger-rate bound, ROI numbers, and spot numbers. It is also written to CSV.

In [ ]:
table = ht.summary_table(pairs, spot=SPOT, threshold=HIT_THR)
display(table)
ht.FIG_DIR.mkdir(parents=True, exist_ok=True)
table.to_csv(ht.FIG_DIR / "summary_all_runs.csv", index=False)
with_dark = [(r, d) for r, d in pairs if d is not None]
if with_dark:
    ht.plot_compare(with_dark, spot=SPOT, save_as="side_by_side_all")

## 5. Delay scans, ladder steps and width series

This rebuilds:
- the delay-scan plots from runs labelled `dscan_...` and `coarse_...`,
- every halving-ladder step from its early and late pair,
- the gate-width series from `wseries_...` runs.

In [ ]:
for prefix in ("dscan", "coarse"):
    scan = [r for r in runs if r["meta"]["settings"]["label"].startswith(prefix + "_")]
    if scan:
        ht.plot_delay_scan(scan, threshold=HIT_THR, spot=SPOT, save_as=f"{prefix}_replot")

for a in [r for r in runs if r["meta"]["settings"]["label"].endswith("_early")]:
    late = a["meta"]["settings"]["label"].replace("_early", "_late")
    b = next((r for r in runs if r["meta"]["settings"]["label"] == late
              and r["meta"]["started"] >= a["meta"]["started"]), None)
    if b is not None:
        ht.compare_halves(a, b, threshold=HIT_THR, spot=SPOT)

ws_sig = [r for r in runs if r["meta"]["settings"]["label"].startswith("wseries_")
          and r["meta"]["settings"]["kind"] == "signal"]
ws_dark = [r for r in runs if r["meta"]["settings"]["label"].startswith("wseries_blocked")]
if ws_sig:
    ht.plot_width_series(ws_sig, threshold=HIT_THR, spot=SPOT, dark_runs=ws_dark, save_as="width_series_replot")